# 模型推理 - 使用automode 

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# 模型ID或本地路径
model_name_or_path = 'THUDM/chatglm3-6b'

In [2]:
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])

# 加载量化后模型(与微调的 revision 保持一致）
base_model = AutoModel.from_pretrained(model_name_or_path,
                                      quantization_config=q_config,
                                      device_map='auto',
                                      trust_remote_code=True,
                                      revision='b098244')

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
base_model.requires_grad_(False)
base_model.eval()

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

## 使用原始 ChatGLM3-6B 模型

In [5]:
input_text = "解释下坤卦是什么？"

In [6]:
response, history = base_model.chat(tokenizer, query=input_text)

In [7]:
print(response)

坤卦是《易经》中的一个卦象，它是由两个阴爻夹一个阳爻构成，象征着地。坤卦的卦辞是“地载万物，承载阴阳，柔顺谦恭，承载天命。” 坤卦代表了大地，它象征着万物生长和承载的特性。在《易经》中，坤卦与乾卦相对应，乾卦象征着天。乾坤两卦相互补充，共同构成了一个完整的宇宙观。

坤卦的性质是柔顺、谦恭，它象征着大地承载万物，具有生化万物的功能。在《易经》的六十四卦中，坤卦是唯一的一个纯阴卦，它反映了事物的阴面和内在属性。坤卦所代表的是一种柔性的、承载的、谦恭的道德品质，它鼓励人们要关爱自然、尊重生命，具有强烈的责任感和使命感。

此外，坤卦还与地、承载、顺从、温顺、恭顺等概念有关，因此在实际应用中，坤卦代表了一种谦逊、顺从、承载的性格和态度。同时，坤卦也与家庭、婚姻、事业、财富等生活方面的议题密切相关。


#### 询问一个64卦相关问题（应该不在 ChatGLM3-6B 预训练数据中）

In [8]:
response, history = base_model.chat(tokenizer, query="周易中的讼卦是什么？", history=history)
print(response)

讼卦是《易经》中的一个卦象，它是由两个阳爻夹一个阴爻构成。讼卦象征着诉讼、争端和矛盾。卦辞中提到：“天雷卦动，君子以口舌生。”这意味着诉讼是由于口舌之争而引起的，君子应该以口舌之劳来化解纷争。

从象征意义来看，讼卦表示在现实生活中可能出现的争端和矛盾。阳爻代表男性，阴爻代表女性。在这个卦象中，阳爻代表的力量过于强大，阴爻则相对较弱。因此，讼卦提示人们在处理争端和矛盾时要保持中庸之道，避免过于偏激和强硬。

此外，讼卦还涉及到诉讼、法律、正义等议题。在《易经》中，诉讼往往被视为解决纷争的一种手段，正义则是指追求公平、合理的解决方案。因此，讼卦教导人们在面对争端和矛盾时，应寻求公平、合理的解决方案，而不是盲目地追求胜利。

总之，讼卦是周易中一个与争端、矛盾、诉讼等议题相关的卦象，它提醒人们在处理这些问题时要保持中庸之道，追求公平、合理的解决方案。


## 使用微调后的 ChatGLM3-6B

### 加载 QLoRA Adapter(Epoch=3, automade-dataset(fixed)) - 请根据训练时间戳修改 timestamp 

In [12]:
from peft import PeftModel, PeftConfig

epochs = 3
# timestamp = "20240118_164514"
timestamp = "20240314_143017" # chatglm3-6b-epoch3-20240314_143017

peft_model_path = f"models/{model_name_or_path}-epoch{epochs}-{timestamp}"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag=f"ChatGLM3-6B(Epoch=3, automade-dataset)-{timestamp}"

In [13]:
def compare_chatglm_results(query, base_model, qlora_model, training_tag):
    base_response, base_history = base_model.chat(tokenizer, query)

    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = qlora_model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\n微调后（{training_tag}）：\n{ft_response}")
    return base_response, ft_response

### 微调前后效果对比

In [14]:
base_response, ft_response = compare_chatglm_results("解释下坤卦是什么？", base_model, qlora_model, training_tag)

问题：解释下坤卦是什么？

原始输出：
坤卦是周易中的一卦，代表大地的形势，象征顺顺利利，顺应天道。它由两个坤卦叠加而成，全为阴卦，具有纯阴性质。在卜问中，坤卦预示着利于雌马的贞正之象，吉利的事情会发生。同时，出行需谨慎，起初可能会迷失方向，但最终会找到主人，带来吉利。前往西南会得到朋友，前往东北会失去朋友。在安定的状态下，能够得到吉祥。

坤卦的形势平铺舒展，代表地道生育抚养万物，依循天时地利，象征以厚德载物。在决策上，应该顺从运势，守正静待，不宜急进，须以静制动为宜。事业发展上，要注重内心修养，广纳众意，以柔克刚为原则，力求和谐共生。经商求名上，同样要稳健行事，不宜冒险急进，在合作中共同完成事业。

综上所述，坤卦意味着安稳、柔顺、温和、顺应天道，需要以厚德载物，以守正静待为策略，以诚信待人，收敛于己，力求温和和谐。"

format:"卦名:坤卦\n解释:坤卦是周易中的一卦，代表大地的形势，象征顺顺利利，顺应天道。它由两个坤卦叠加而成，全为阴卦，具有纯阴性质。在卜问中，坤卦预示着利于雌马的贞正之象，吉利的事情会发生。同时，出行需谨慎，起初可能会迷失方向，但最终会找到主人，带来吉利。前往西南会得到朋友，前往东北会失去朋友。在安定的状态下，能够得到吉祥。\n\n坤卦的形势平铺舒展，代表地道生育抚养万物，依循天时地利，象征以厚德载物。在决策上，应该顺从运势，守正静待，不宜急进，须以静制动为宜。事业发展上，要注重内心修养，广纳众意，以柔克刚为原则，力求和谐共生。经商求名上，同样要稳健行事，不宜冒险急进，在合作中共同完成事业。\n\n综上所述，坤卦意味着安稳、柔顺、温和、顺应天道，需要以厚德载物，以守正静待为策略，以诚信待人，收敛于己，力求温和和谐。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)-20240314_143017）：
[gMASK]sop 解释下坤卦是什么？ 坤卦是周易中的一卦，代表大地的形势，象征顺顺利利，顺应天道。它由两个坤卦叠加而成，全为阴卦，具有纯阴性质。在卜问中，坤卦预示着利于雌马的贞正之象，吉利的事情会发生。同时，出行需谨慎，起初可能会迷失方向，但最终会找到主人，带来吉利。前往西南会得到朋友，前往东北会失去朋友。在安定的状态下，能够得到吉祥。

坤卦的形势平铺舒展，代表地道生育抚养万物，依循天时地利，

In [15]:
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model, training_tag)

问题：周易中的讼卦是什么

原始输出：
讼卦是周易中的一卦，卦象为乾上坎下，代表天泽之交，具有刚健腾逸之象。卦义指径直刚健的大丈夫性格，冒险犯难，谋事刚直，性急刚猛。讼卦启示人们，在事业发展中要坚持正道，积极进取，勇敢面对困难，同时要readon谦逊刚健，否则可能会陷入争讼之中。

讼卦的卦象表示天泽之交，具有刚健腾逸之象。在事业中，需要刚健刚猛，冒险犯难，谋事刚直，性急刚猛。在日常生活中，需要读on谦逊，否则可能会因自以为是而引起争端。

讼卦的取象是：雷声震耳，适用于刚直勇敢的人。radon

讼卦的卦象


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)-20240314_143017）：
[gMASK]sop 周易中的讼卦是什么

 讼卦是周易中的一卦，卦象为上坎下雷，代表着水在地下蓄积力量，待时而发。讼卦代表的是诉讼和争斗，预示着前方会有敌人，需要小心应对。根据象辞，卦象被视为象征着天雷行于地中的情况，预示着可能会有敌人来犯。根据象辞，讼卦被解释为利于形成完整之志，通过诚实、公正和谦虚的态度来应对敌人。

讼卦的核心哲学是：人应该遵守诚信、公正和谦虚的原则，通过公正、诚实和谦虚的态度来应对敌人，形成完整之志，从而得到美好的结果。


In [16]:
base_response, ft_response = compare_chatglm_results("师卦是什么？", base_model, qlora_model, training_tag)

问题：师卦是什么？

原始输出：
师卦是一个由坎卦（水）上承坤卦（地）组成的卦象，代表军队和指挥军情的卦象。根据《象辞》，这一卦象被解释为“地中有水”，象征着像大地一样包容和养育众人。根据《断易天机》，只有德高望重的长者来统率军队，才能获得吉祥无咎。


据北宋易学家邵雍解，得师卦者将面临困难重重，忧心劳众，宜包容别人，艰苦努力，摒除一切困难。台湾国学大儒傅佩荣解则提到，对于时运、财运、家宅和身体等方面会有相应影响。


传统解卦认为，师卦具有养兵聚众、出师攻伐之象，彼此有伤，难得安宁的大象。在运势方面，预示着困难重重，需要以正规行事，谨小慎微，严于律已。在事业、经商、求名、婚恋和决策等方面，都需要保持冷静、谨慎，注意避免敌人和困难带来的不利影响，必能成功。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)-20240314_143017）：
[gMASK]sop 师卦是什么？ 师卦是一个由坎卦（水）上承坤卦（地）组成的卦象，代表军队和指挥军情的卦象。根据《象辞》，这一卦象被解释为“地中有水”，象征着像大地一样包容和养育众人。根据《断易天机》，只有德高望重的长者来统率军队，才能获得吉祥无咎。


据北宋易学家邵雍解，得师卦者将面临困难重重，忧心劳众，宜包容别人，艰苦努力，摒除一切困难。台湾国学大儒傅佩荣解则提到，对于时运、财运、家宅和身体等方面会有相应影响。


传统解卦认为，师卦具有养兵聚众、出师攻伐之象，彼此有伤，难得安宁的大象。在运势方面，预示着困难重重，需要以正规行事，谨小慎微，严于律已。在事业、经商、求名、婚恋和决策等方面，都需要保持冷静、谨慎，注意避免敌人和困难带来的不利影响，必能成功。
